In [27]:
import pandas as pd
import glob
import os

# --- CONFIG ---
# Master Excel file (on Desktop)
master_path = "/Users/EthanMcElhone/Desktop/Master.xlsx"

# Folder containing all borough CSVs
borough_folder = "/Users/EthanMcElhone/Desktop/Borough"

# Output Excel file (Desktop)
output_file = "/Users/EthanMcElhone/Desktop/master_with_borough_data.xlsx"

# Standard columns for merged data
standard_cols = ['transactionid', 'priceper', 'year', 'dateoftransfer', 'postcode', 'propertytype']

# --- CHECK FOLDER EXISTS ---
if not os.path.exists(borough_folder):
    raise FileNotFoundError(f"Borough folder not found: {borough_folder}")

# --- LOAD MASTER FILE ---
master_df = pd.read_excel(master_path, dtype=str)

# Rename transactionid column in master by position (F = 6th column, index 5)
master_df.rename(columns={master_df.columns[5]: "transactionid"}, inplace=True)
master_df["transactionid"] = master_df["transactionid"].astype(str)

# --- FIND CSV FILES ---
csv_files = glob.glob(os.path.join(borough_folder, "*.csv")) + glob.glob(os.path.join(borough_folder, "*.CSV"))
if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in folder: {borough_folder}")
print(f"Found {len(csv_files)} CSV files.")

# --- PROCESS BOROUGH CSV FILES ---
borough_data = []

for file in csv_files:
    df = pd.read_csv(file, dtype=str)

    # Rename transactionid column by position (I = 9th column, index 8)
    df.rename(columns={df.columns[8]: "transactionid"}, inplace=True)
    df["transactionid"] = df["transactionid"].astype(str)

    # Normalize headers: lowercase, remove spaces and non-breaking spaces
    df.columns = df.columns.str.lower().str.replace(" ", "").str.replace("\xa0", "")

    # Map columns to standard columns
    col_map = {
        "pricepaid": "priceper",
        "year": "year",
        "dateoftransfer": "dateoftransfer",
        "postcode": "postcode",
        "propertytype": "propertytype"
    }
    existing_map = {k: v for k, v in col_map.items() if k in df.columns}
    df.rename(columns=existing_map, inplace=True)

    # Keep only standard columns + transactionid
    available_cols = ["transactionid"] + [c for c in standard_cols if c in df.columns and c != "transactionid"]
    borough_data.append(df[available_cols])

# --- CONCAT ALL BOROUGH DATA ---
borough_df = pd.concat(borough_data, ignore_index=True)

# --- MERGE WITH MASTER ---
merged_df = master_df.merge(borough_df, on="transactionid", how="left")

# --- SAVE OUTPUT ---
merged_df.to_excel(output_file, index=False)
print(f"Merge complete! Saved as: {output_file}")


Found 34 CSV files.
Merge complete! Saved as: /Users/EthanMcElhone/Desktop/master_with_borough_data.xlsx


In [35]:
import pandas as pd
import glob
import os

# --- CONFIG ---
master_path = "/Users/EthanMcElhone/Desktop/Master.xlsx"
borough_folder = "/Users/EthanMcElhone/Desktop/Borough"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_Borough.xlsx"

# --- LOAD MASTER FILE ---
master_df = pd.read_excel(master_path, dtype=str)

# Rename transactionid column by position (F = 6th column, index 5)
master_df.rename(columns={master_df.columns[5]: "transactionid"}, inplace=True)
master_df["transactionid"] = master_df["transactionid"].astype(str)

# --- PROCESS BOROUGH CSV FILES TO GET MAPPING ---
csv_files = glob.glob(os.path.join(borough_folder, "*.csv")) + glob.glob(os.path.join(borough_folder, "*.CSV"))
if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in folder: {borough_folder}")

borough_mapping = []

for file in csv_files:
    df = pd.read_csv(file, dtype=str)

    # Rename transactionid column by position (I = 9th column, index 8)
    df.rename(columns={df.columns[8]: "transactionid"}, inplace=True)
    df["transactionid"] = df["transactionid"].astype(str)

    # Extract borough name from filename
    borough_name = os.path.basename(file).split("_link_")[0]
    borough_name = os.path.splitext(borough_name)[0]  

    # Keep only transactionid + borough
    df_borough = pd.DataFrame({
        "transactionid": df["transactionid"],
        "borough": borough_name
    })

    borough_mapping.append(df_borough)

# Combine all borough mappings
borough_df = pd.concat(borough_mapping, ignore_index=True)

# --- MERGE WITH MASTER ---
merged_df = master_df.merge(borough_df, on="transactionid", how="left")

# --- SAVE OUTPUT ---
merged_df.to_excel(output_file, index=False)
print(f"Master file updated with borough column! Saved as: {output_file}")


Master file updated with borough column! Saved as: /Users/EthanMcElhone/Desktop/Master_with_Borough.xlsx


In [ ]:
import pandas as pd
import os

# --- CONFIG ---
master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
cpih_file = "/Users/EthanMcElhone/Downloads/CPIH_inflation_statistics.csv"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_CPIH_Lagged.xlsx"

# --- LOAD MASTER FILE ---
master_df = pd.read_excel(master_file, dtype=str)

# Ensure dateoftransfer exists
if 'dateoftransfer' not in master_df.columns:
    raise ValueError("Master file must have a 'dateoftransfer' column.")

# Convert transaction date to datetime
master_df['dateoftransfer'] = pd.to_datetime(master_df['dateoftransfer'], errors='coerce')

# Extract year and month
master_df['year'] = master_df['dateoftransfer'].dt.year.astype(int)
master_df['month'] = master_df['dateoftransfer'].dt.month

# Determine quarter
def month_to_quarter(month):
    if month in [1,2,3]: return "Q1"
    elif month in [4,5,6]: return "Q2"
    elif month in [7,8,9]: return "Q3"
    elif month in [10,11,12]: return "Q4"
    else: return None

master_df['quarter'] = master_df['month'].apply(month_to_quarter)

# Create lagged year for CPIH
master_df['lagged_year'] = master_df['year'] - 1
master_df['lagged_year_quarter'] = master_df['lagged_year'].astype(str) + " " + master_df['quarter']

# --- LOAD CPIH DATA ---
cpih_df = pd.read_csv(cpih_file, dtype=str)
cpih_df.rename(columns={'year':'year_quarter'}, inplace=True)  # year column contains "YYYY QX"

# Merge only lagged CPIH rate
merged_df = master_df.merge(
    cpih_df[['year_quarter','cpih_rate']], 
    left_on='lagged_year_quarter', right_on='year_quarter', how='left'
)

# Clean up columns and rename CPIH rate
merged_df = merged_df.drop(columns=['year_quarter','year','month','quarter','lagged_year','lagged_year_quarter'])
merged_df.rename(columns={'cpih_rate':'cpih_rate_lagged_1yr'}, inplace=True)

# Format date to remove 00:00:00
merged_df['dateoftransfer'] = merged_df['dateoftransfer'].dt.strftime('%Y-%m-%d')


# --- SAVE FINAL OUTPUT ---
merged_df.to_excel(output_file, index=False)
print(f"Master file updated with 1-year lagged CPIH rate! Saved as: {output_file}")


/var/folders/x9/z2rcxv3j2yb764n0kvtclgsr0000gn/T/ipykernel_99629/265565432.py:17: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  master_df['dateoftransfer'] = pd.to_datetime(master_df['dateoftransfer'], errors='coerce')


Master file updated with 1-year lagged CPIH rate! Saved as: /Users/EthanMcElhone/Desktop/Master_with_CPIH_Lagged.xlsx


In [53]:
import pandas as pd

# --- LOAD MASTER FILE ---
master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
master_df = pd.read_excel(master_file, dtype=str)

# Ensure dateoftransfer exists
if 'dateoftransfer' not in master_df.columns:
    raise ValueError("Master file must have a 'dateoftransfer' column.")

# Convert dateoftransfer to datetime
master_df['dateoftransfer'] = pd.to_datetime(master_df['dateoftransfer'], errors='coerce')

# Extract year
master_df['year'] = master_df['dateoftransfer'].dt.year.astype(str)

# Format dateoftransfer to remove 00:00:00
master_df['dateoftransfer'] = master_df['dateoftransfer'].dt.strftime('%Y-%m-%d')

# --- SAVE UPDATED MASTER ---
output_file = "/Users/EthanMcElhone/Desktop/Master_with_Year.xlsx"
master_df.to_excel(output_file, index=False)
print(f"Master file updated with 'year' column and cleaned date! Saved as: {output_file}")


Master file updated with 'year' column and cleaned date! Saved as: /Users/EthanMcElhone/Desktop/Master_with_Year.xlsx


In [ ]:
import pandas as pd
import os

# --- CONFIG ---
master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
crime_file = "/Users/EthanMcElhone/Desktop/crime.csv"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_Crime_Lagged.xlsx"

# --- LOAD MASTER FILE ---
master_df = pd.read_excel(master_file, dtype=str)

# Ensure master file has 'borough' and 'dateoftransfer' columns
if 'borough' not in master_df.columns or 'dateoftransfer' not in master_df.columns:
    raise ValueError("Master file must have 'borough' and 'dateoftransfer' columns.")

# Convert transaction date to datetime and extract year
master_df['dateoftransfer'] = pd.to_datetime(master_df['dateoftransfer'], errors='coerce')
master_df['year'] = master_df['dateoftransfer'].dt.year.astype(int)

# --- LOAD CRIME DATA ---
crime_df = pd.read_csv(crime_file, dtype=str)

# Ensure crime_df has required columns
if not {'borough','year','crime'}.issubset(crime_df.columns):
    raise ValueError("Crime CSV must have 'borough', 'year', and 'crime' columns.")

# Convert year to int for lagging
crime_df['year'] = crime_df['year'].astype(int)

# --- CREATE LAGGED YEAR IN MASTER ---
master_df['lagged_year'] = master_df['year'] - 1

# --- MERGE LAGGED CRIME RATE ---
merged_df = master_df.merge(
    crime_df[['borough','year','crime']],
    left_on=['borough','lagged_year'],
    right_on=['borough','year'],
    how='left'
)

# Drop duplicate 'year' from crime_df and rename column
merged_df = merged_df.drop(columns=['year_y'])
merged_df.rename(columns={'year_x':'year','crime':'crime_lagged_1yr'}, inplace=True)

# --- SAVE FINAL OUTPUT ---
merged_df.to_excel(output_file, index=False)
print(f"Master file updated with 1-year lagged crime rate! Saved as: {output_file}")


Master file updated with 1-year lagged crime rate! Saved as: /Users/EthanMcElhone/Desktop/Master_with_Crime_Lagged.xlsx


In [58]:
import pandas as pd

# --- CONFIG ---
master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
crime_file = "/Users/EthanMcElhone/Desktop/crime.csv"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_Crime.xlsx"

# --- LOAD MASTER FILE ---
master_df = pd.read_excel(master_file, dtype=str)

# Ensure necessary columns
if 'borough' not in master_df.columns or 'dateoftransfer' not in master_df.columns:
    raise ValueError("Master file must have 'borough' and 'dateoftransfer' columns.")

# --- EXTRACT YEAR FROM dateoftransfer AS STRING ---
master_df['year'] = master_df['dateoftransfer'].str[:4]  # just take first 4 chars

# --- LOAD CRIME DATA ---
crime_df = pd.read_csv(crime_file, dtype=str)

# normalise borough names
master_df['borough'] = master_df['borough'].str.strip().str.lower()
crime_df['borough'] = crime_df['borough'].str.strip().str.lower()

# Ensure crime_df has required columns
if not {'borough','year','crime'}.issubset(crime_df.columns):
    raise ValueError("Crime CSV must have 'borough', 'year', and 'crime' columns.")

# --- MERGE SAME-YEAR CRIME DATA ---
merged_df = master_df.merge(
    crime_df[['borough', 'year', 'crime']],
    on=['borough', 'year'],
    how='left'
)

# Rename crime column
merged_df.rename(columns={'crime': 'crime_same_year'}, inplace=True)

# --- SAVE OUTPUT ---
merged_df.to_excel(output_file, index=False)
print(f"Master file updated with same-year crime rate! Saved as: {output_file}")


Master file updated with same-year crime rate! Saved as: /Users/EthanMcElhone/Desktop/Master_with_Crime.xlsx


In [60]:
import pandas as pd

# --- CONFIG ---
master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
ptal_file = "/Users/EthanMcElhone/Downloads/PTAL.csv"  
output_file = "/Users/EthanMcElhone/Desktop/Master_with_PTAL.xlsx"

# --- LOAD MASTER ---
master_df = pd.read_excel(master_file, dtype=str)

# Ensure master has 'postcode'
if 'postcode' not in master_df.columns:
    raise ValueError("Master file must contain a 'postcode' column.")

# --- LOAD PTAL FILE ---
ptal_df = pd.read_csv(ptal_file, dtype=str)

# Ensure ptal file has 'postcode' and 'ptal'
if not {'postcode', 'ptal'}.issubset(ptal_df.columns):
    raise ValueError("PTAL file must contain 'postcode' and 'ptal' columns.")

# --- CLEAN POSTCODES FOR MATCHING ---
master_df['postcode_clean'] = master_df['postcode'].str.replace(" ", "").str.lower()
ptal_df['postcode_clean'] = ptal_df['postcode'].str.replace(" ", "").str.lower()

# --- MERGE ---
merged_df = master_df.merge(
    ptal_df[['postcode_clean', 'ptal']],
    on='postcode_clean',
    how='left'
)

# --- CLEAN UP ---
merged_df.drop(columns=['postcode_clean'], inplace=True)

# --- SAVE ---
merged_df.to_excel(output_file, index=False)

print(f"PTAL merged successfully! Saved to: {output_file}")


PTAL merged successfully! Saved to: /Users/EthanMcElhone/Desktop/Master_with_PTAL.xlsx


In [ ]:
### PROBLEM WITH THIS ###

import pandas as pd

# --- CONFIG ---
master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
summary_file = "/Users/EthanMcElhone/Downloads/property_summary_year_borough_areabin.csv"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_lagged_ppsqm.csv"

# --- LOAD MASTER FILE ---
master_df = pd.read_excel(master_file, dtype=str)

# Check required columns
if 'postcode' not in master_df.columns or 'dateoftransfer' not in master_df.columns:
    raise ValueError("Master file must contain 'postcode' and 'dateoftransfer' columns.")

# --- CLEAN + EXTRACT OUTWARD POSTCODE ---
master_df['outward'] = (
    master_df['postcode']
    .str.strip()
    .str.extract(r"^([A-Za-z]{1,2}\d[A-Za-z\d]?)")[0]
)

# --- EXTRACT YEAR ---
master_df['year'] = master_df['dateoftransfer'].str[:4].astype(int)

# Create lagged year
master_df['lagged_year'] = master_df['year'] - 1

# --- LOAD SUMMARY FILE ---
summary_df = pd.read_csv(summary_file, dtype=str)

required_cols = {'year', 'postcode', 'average_price', 'average_area'}
if not required_cols.issubset(summary_df.columns):
    raise ValueError(f"Summary CSV missing required columns: {required_cols}")

summary_df['year'] = summary_df['year'].astype(int)
summary_df['average_price'] = summary_df['average_price'].astype(float)
summary_df['average_area'] = summary_df['average_area'].astype(float)

summary_df['outward'] = summary_df['postcode'].str.replace(r"^\d+_", "", regex=True)

summary_df['price_per_sqm'] = summary_df['average_price'] / summary_df['average_area']

summary_df = summary_df[['year', 'outward', 'price_per_sqm']]

# --- MERGE USING LAGGED YEAR ---
merged_df = master_df.merge(
    summary_df,
    left_on=['outward', 'lagged_year'],
    right_on=['outward', 'year'],
    how='left'
)

merged_df.rename(columns={'price_per_sqm': 'lagged_price_per_sqm'}, inplace=True)

merged_df = merged_df.drop(columns=['year_y'])
merged_df.rename(columns={'year_x': 'year'}, inplace=True)

# --- NEW FIX: FILL NANs BY GROUP MEAN ---
merged_df['lagged_price_per_sqm'] = (
    merged_df.groupby(['outward', 'lagged_year'])['lagged_price_per_sqm']
             .transform(lambda x: x.fillna(x.mean()))
)

# --- SAVE ---
merged_df.to_csv(output_file, index=False)

print(f"Done! CSV saved to: {output_file}")


Done! CSV saved to: /Users/EthanMcElhone/Desktop/Master_with_lagged_ppsqm.csv


In [74]:
import glob
borough_folder =  "/Users/EthanMcElhone/Desktop/Borough/"  # adjust path if needed
csv_files = glob.glob(os.path.join(borough_folder, "*.csv")) + glob.glob(os.path.join(borough_folder, "*.CSV"))
print("Files found:", csv_files)


Files found: ['/Users/EthanMcElhone/Desktop/Borough/Croydon.csv', '/Users/EthanMcElhone/Desktop/Borough/Redbridge.csv', '/Users/EthanMcElhone/Desktop/Borough/Waltham Forest.csv', '/Users/EthanMcElhone/Desktop/Borough/Hammersmith and Fulham.csv', '/Users/EthanMcElhone/Desktop/Borough/Brent.csv', '/Users/EthanMcElhone/Desktop/Borough/Lewisham.csv', '/Users/EthanMcElhone/Desktop/Borough/Merton.csv', '/Users/EthanMcElhone/Desktop/Borough/Tower Hamlets.csv', '/Users/EthanMcElhone/Desktop/Borough/Camden.csv', '/Users/EthanMcElhone/Desktop/Borough/Westminster.csv', '/Users/EthanMcElhone/Desktop/Borough/Hackney.csv', '/Users/EthanMcElhone/Desktop/Borough/Greenwich.csv', '/Users/EthanMcElhone/Desktop/Borough/Havering.csv', '/Users/EthanMcElhone/Desktop/Borough/Haringey.csv', '/Users/EthanMcElhone/Desktop/Borough/Harrow.csv', '/Users/EthanMcElhone/Desktop/Borough/Southwark.csv', '/Users/EthanMcElhone/Desktop/Borough/Richmond upon Thames.csv', '/Users/EthanMcElhone/Desktop/Borough/Hounslow.csv', 